# NER - BERT Fine-Tuning

## Objective

This notebook trains BERT for Named Entity Recognition (NER) using the CoNLL-2003 dataset.

Two approaches are evaluated:

- Full Fine-Tuning: all BERT parameters are trainable.
- Feature-Based: BERT parameters are frozen and only the classification head is trained.

The base model is `bert-base-uncased`.

A fixed random seed of 42 is used for reproducibility.

In [ ]:
!pip install -U transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.0 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0


In [ ]:
SEED = 42

import os
import random
import numpy as np
import torch
from transformers import set_seed

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

print("Seed configurada:", SEED)

Seed configurada: 42


## Dataset: CoNLL-2003

The CoNLL-2003 dataset is used for Named Entity Recognition.

The dataset contains sentences with token-level entity labels.

The main entity categories include:

- Person (PER)
- Organization (ORG)
- Location (LOC)
- Miscellaneous (MISC)
- Other (O)

Each token is assigned a label that indicates whether it belongs to an entity and its position within that entity.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("lhoestq/conll2003")

print(dataset)

dataset_infos.json:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.07MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


## Dataset Preparation

The dataset is divided into training, validation, and test sets.

The original token-level labels are converted into numerical IDs so that they can be used by the BERT token classification model.

In [ ]:
ner_feature = dataset["train"].features["ner_tags"]

print("Tipo de ner_tags:", ner_feature)

Tipo de ner_tags: List(Value('int64'))


In [ ]:
print("Primeras etiquetas:", dataset["train"][0]["ner_tags"])

Primeras etiquetas: [3, 0, 7, 0, 0, 0, 7, 0, 0]


In [ ]:
label_names = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC",
    "B-MISC",
    "I-MISC"
]

print("Número de etiquetas:", len(label_names))
print("Etiquetas:", label_names)

Número de etiquetas: 9
Etiquetas: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


## NER Label Mapping

The model uses nine labels:

- `O`: Outside an entity
- `B-PER`: Beginning of a person entity
- `I-PER`: Inside a person entity
- `B-ORG`: Beginning of an organization entity
- `I-ORG`: Inside an organization entity
- `B-LOC`: Beginning of a location entity
- `I-LOC`: Inside a location entity
- `B-MISC`: Beginning of a miscellaneous entity
- `I-MISC`: Inside a miscellaneous entity

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer cargado correctamente.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer cargado correctamente.


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

## Tokenization and Label Alignment

The BERT tokenizer divides text into subword tokens when necessary.

For NER, labels must be aligned with the resulting tokens.

The label of the first subword is kept, while additional subwords are ignored during loss calculation using the label `-100`.

This prevents the same original token from being counted multiple times.

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})


In [ ]:
print(tokenized_dataset["train"][0])

{'input_ids': [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, -100]}


In [ ]:
from transformers import AutoModelForTokenClassification

num_labels = len(label_names)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={i: label for i, label in enumerate(label_names)},
    label2id={label: i for i, label in enumerate(label_names)}
)

print("Modelo BERT para NER cargado correctamente.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

Modelo BERT para NER cargado correctamente.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        for p, l in zip(prediction, label):
            if l != -100:
                true_predictions.append(p)
                true_labels.append(l)

    accuracy = accuracy_score(
        true_labels,
        true_predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Métricas preparadas.")

Métricas preparadas.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_ner",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    seed=SEED,
    data_seed=SEED,

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

print("Configuración de entrenamiento preparada.")

Configuración de entrenamiento preparada.


In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

print("Data collator preparado.")

Data collator preparado.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("Trainer preparado.")

Trainer preparado.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.061513,0.051327,0.986171,0.986385,0.986171,0.986071
2,0.035635,0.046083,0.988353,0.988389,0.988353,0.988360
3,0.017277,0.047221,0.988431,0.988403,0.988431,0.988411


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=2634, training_loss=0.06303986723710844, metrics={'train_runtime': 453.7126, 'train_samples_per_second': 92.841, 'train_steps_per_second': 5.805, 'total_flos': 1019261176981398.0, 'train_loss': 0.06303986723710844, 'epoch': 3.0})

## Full Fine-Tuning

In the Full Fine-Tuning approach, all BERT parameters are trainable.

The model can therefore adapt its pretrained representations to the NER task.

The learning rate used for this approach is `2e-5`.

In [ ]:
results = trainer.evaluate()

print("Accuracy:", results["eval_accuracy"])
print("Precision:", results["eval_precision"])
print("Recall:", results["eval_recall"])
print("F1:", results["eval_f1"])

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.017277,0.047221,3,0.988431,0.988403,0.988431,0.988411


Accuracy: 0.9884307500535613
Precision: 0.9884027215892549
Recall: 0.9884307500535613
F1: 0.9884107525954072


In [ ]:
MODEL_DIR = "./models/conll2003_bert_ner"

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("Modelo guardado en:", MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en: ./models/conll2003_bert_ner


In [ ]:
import os

print(os.listdir(MODEL_DIR))

['training_args.bin', 'model.safetensors', 'config.json', 'tokenizer_config.json', 'tokenizer.json']


## Feature-Based Training

In the Feature-Based approach, the pretrained BERT parameters are frozen.

Only the task-specific classification layer is trained.

This approach uses BERT as a fixed feature extractor.

The learning rate used for the classification head is `1e-3`.

In [ ]:
from transformers import AutoModelForTokenClassification

MODEL_NAME = "bert-base-uncased"

model_feature = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label={i: label for i, label in enumerate(label_names)},
    label2id={label: i for i, label in enumerate(label_names)}
)

print("BERT cargado para Feature-based.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

BERT cargado para Feature-based.


In [ ]:
# Congelar todas las capas de BERT
for param in model_feature.base_model.parameters():
    param.requires_grad = False

# La capa de clasificación SÍ se puede entrenar
for param in model_feature.classifier.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model_feature.parameters() if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_feature.parameters()
)

print("Parámetros totales:", total_params)
print("Parámetros entrenables:", trainable_params)
print("BERT congelado correctamente.")

Parámetros totales: 108898569
Parámetros entrenables: 6921
BERT congelado correctamente.


In [ ]:
from transformers import TrainingArguments

training_args_feature = TrainingArguments(
    output_dir="./results_ner_feature",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

print("Configuración Feature-based preparada.")

Configuración Feature-based preparada.


In [ ]:
from transformers import Trainer

trainer_feature = Trainer(
    model=model_feature,
    args=training_args_feature,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer Feature-based preparado.")

Trainer Feature-based preparado.


In [ ]:
trainer_feature.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.159370,0.138045,0.965409,0.963519,0.965409,0.963087
2,0.145372,0.121944,0.969149,0.967461,0.969149,0.967595
3,0.126230,0.118002,0.970220,0.968688,0.970220,0.968871


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=2634, training_loss=0.1690673333908811, metrics={'train_runtime': 178.0628, 'train_samples_per_second': 236.563, 'train_steps_per_second': 14.793, 'total_flos': 1019261176981398.0, 'train_loss': 0.1690673333908811, 'epoch': 3.0})

In [ ]:
results_feature = trainer_feature.evaluate()

print("NER Feature-based")
print("Accuracy:", results_feature["eval_accuracy"])
print("Precision:", results_feature["eval_precision"])
print("Recall:", results_feature["eval_recall"])
print("F1:", results_feature["eval_f1"])

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.126230,0.118002,3,0.970220,0.968688,0.970220,0.968871


NER Feature-based
Accuracy: 0.9702198936563894
Precision: 0.9686882081469317
Recall: 0.9702198936563894
F1: 0.9688711698433619


## Evaluation

The NER models are evaluated using token-level metrics.

The evaluation includes:

- Accuracy
- Precision
- Recall
- F1 Score

The metrics are calculated using weighted averages across the token labels.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/BERT_FineTuning_Project"

!find "$BASE_DIR/models" -maxdepth 2 -type f | grep -i "ner"

Mounted at /content/drive
find: ‘/content/drive/MyDrive/BERT_FineTuning_Project/models’: No such file or directory


In [ ]:
!find "$BASE_DIR/checkpoints" -maxdepth 3 -type f | grep -i "ner"

find: ‘/content/drive/MyDrive/BERT_FineTuning_Project/checkpoints’: No such file or directory


In [ ]:
MODEL_DIR_FULL = f"{BASE_DIR}/models/conll2003_bert_ner_full"

trainer.save_model(MODEL_DIR_FULL)
tokenizer.save_pretrained(MODEL_DIR_FULL)

print("NER Full guardado en Drive:")
print(MODEL_DIR_FULL)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NER Full guardado en Drive:
/content/drive/MyDrive/BERT_FineTuning_Project/models/conll2003_bert_ner_full


In [ ]:
!ls -lah "$BASE_DIR/models/conll2003_bert_ner_full"

total 417M
drwx------ 2 root root 4.0K Sep 25 22:52 .
drwx------ 4 root root 4.0K Sep 25 22:52 ..
-rw------- 1 root root 1.1K Sep 25 22:52 config.json
-rw------- 1 root root 416M Sep 25 22:52 model.safetensors
-rw------- 1 root root  351 Sep 25 22:52 tokenizer_config.json
-rw------- 1 root root 695K Sep 25 22:52 tokenizer.json
-rw------- 1 root root 5.1K Sep 25 22:52 training_args.bin


In [ ]:
MODEL_DIR_FEATURE = f"{BASE_DIR}/models/conll2003_bert_ner_feature"

trainer_feature.save_model(MODEL_DIR_FEATURE)
tokenizer.save_pretrained(MODEL_DIR_FEATURE)

print("NER Feature guardado en Drive:")
print(MODEL_DIR_FEATURE)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NER Feature guardado en Drive:
/content/drive/MyDrive/BERT_FineTuning_Project/models/conll2003_bert_ner_feature


In [ ]:
!ls -lah "$BASE_DIR/models/conll2003_bert_ner_feature"

total 417M
drwx------ 2 root root 4.0K Sep 25 22:52 .
drwx------ 4 root root 4.0K Sep 25 22:52 ..
-rw------- 1 root root 1.1K Sep 25 22:52 config.json
-rw------- 1 root root 416M Sep 25 22:52 model.safetensors
-rw------- 1 root root  351 Sep 25 22:52 tokenizer_config.json
-rw------- 1 root root 695K Sep 25 22:52 tokenizer.json
-rw------- 1 root root 5.1K Sep 25 22:52 training_args.bin


## Final Results

The Full Fine-Tuning approach achieved higher performance than the Feature-Based approach.

### Full Fine-Tuning

- Accuracy: 98.84%
- Precision: 98.84%
- Recall: 98.84%
- F1 Score: 98.84%

### Feature-Based

- Accuracy: 97.02%
- Precision: 96.87%
- Recall: 97.02%
- F1 Score: 96.89%

The results show that allowing BERT to update its pretrained parameters improved performance for the NER task.